In [1]:
"""
Datafile is a text file with one sentence per line _DATASETS/data.txt
tf_gpt2_keras_lora is the name of the fine-tuned model
"""

import tensorflow as tf
from transformers import GPT2Tokenizer, TFGPT2LMHeadModel
from transformers.modeling_tf_utils import get_initializer
import os

In [2]:


# use 2 cores
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)

# Use pretrained model if it exists
# otherwise download it
if os.path.exists("tf_gpt2_keras_lora"):
    print("Model exists")
    # use pretrained model
    model = TFGPT2LMHeadModel.from_pretrained("tf_gpt2_keras_lora")
else:
    print("Downloading model")
    model = TFGPT2LMHeadModel.from_pretrained("gpt2")

# Load the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

w:\Personal Projects\UCPP\ucppvenv\lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\golla\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [3]:
import PyPDF2
import pandas as pd
def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as pdf_file:
        reader = PyPDF2.PdfReader(pdf_file)
        title = reader.metadata.title
        text = ""
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text()
        return title, text

In [4]:
pdf_path = "upsc_full_doc.pdf"
title_1,context_1 = extract_text_from_pdf(pdf_path)

In [5]:
data = {'Title':'UPSC Full Paper', 'Content':context_1}
df = pd.DataFrame(data, index=[0])

corpus = ''.join(df['Content'])

In [6]:

# Encode the data using the tokenizer and truncate the sequences to a maximum length of 1024 tokens
input_ids = []
for line in corpus.split(" "):
    encoding = tokenizer.encode(line, add_special_tokens=True, max_length=1024, truncation=True)
    input_ids.append(encoding)


In [12]:
input_ids[0]

[16]

In [7]:


# Define some params
batch_size = 2
num_epochs = 3
learning_rate = 5e-5

# Define the optimizer and loss function
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Fine-tune the model using low-rank adaptation and attention pruning
for layer in model.transformer.h:
    layer.attention_output_dense = tf.keras.layers.Dense(units=256, kernel_initializer=get_initializer(0.02), name="attention_output_dense")
    
model.summary()

# Train the model
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    
    # Shuffle the input data
    #input_ids = tf.random.shuffle(input_ids)
    
    for i in range(0, len(input_ids), batch_size):
        batch = input_ids[i:i+batch_size]
        # Pad the batch to the same length
        batch = tf.keras.preprocessing.sequence.pad_sequences(batch, padding="post")
        # Define the inputs and targets
        inputs = batch[:, :-1]
        targets = batch[:, 1:]
        # Compute the predictions and loss
        with tf.GradientTape() as tape:
            logits = model(inputs)[0]
            loss = loss_fn(targets, logits)
        # Compute the gradients and update the parameters
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        
        # Print the loss every 10 batches
        if i % (10 * batch_size) == 0:
            print(f"Batch {i}/{len(input_ids)} - loss: {loss:.4f}")
            
# Save the fine-tuned model
model.save_pretrained("tf_gpt2_keras_lora")

# Generate text using the fine-tuned model
input_ids = tokenizer.encode("How much wood", return_tensors="tf")
output = model.generate(input_ids, max_length=100, do_sample=True, top_k=50, top_p=0.95, temperature=0.9)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Model: "tfgpt2lm_head_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 transformer (TFGPT2MainLay  multiple                  124439808 
 er)                                                             
                                                                 
Total params: 124439808 (474.70 MB)
Trainable params: 124439808 (474.70 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/3
Batch 0/59201 - loss: 0.0000
Batch 20/59201 - loss: 2.4156
Batch 40/59201 - loss: 0.0000
Batch 60/59201 - loss: 3.4101
Batch 80/59201 - loss: 3.8264
Batch 100/59201 - loss: 5.0156
Batch 120/59201 - loss: 2.1130
Batch 140/59201 - loss: 4.8779
Batch 160/59201 - loss: 3.7922
Batch 180/59201 - loss: 9.3996
Batch 200/59201 - loss: 0.4318
Batch 220/59201 - loss: 0.0000
Batch 240/59201 - loss: 0.0000
Batch 260/59201 - loss: 4.7409
Batch 280/59201 - loss: 0

KeyboardInterrupt: 

In [15]:

from transformers import BertTokenizer, BertForQuestionAnswering
import torch


In [29]:
# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

# Function to answer a question given a context
def answer_question(question, context):
    inputs = tokenizer.encode_plus(question, context, add_special_tokens=True, return_tensors="pt", truncation=True)
    input_ids = inputs["input_ids"].tolist()[0]

    text_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    answer_start_scores, answer_end_scores = model(**inputs, return_dict=False)

    # Correctly find the positions for the start and end of the answer
    answer_start = torch.argmax(answer_start_scores)  # Removed erroneous `answer and`
    answer_end = torch.argmax(answer_end_scores) + 1  # Removed erroneous `answer and`

    answer = tokenizer.convert_tokens_to_string(text_tokens[answer_start:answer_end])

    return answer

def get_relevant_context(full_text, keyword, window=500):
    start_index = full_text.lower().find(keyword.lower())
    if start_index == -1:
        return "Keyword not found in the text"  # Handle the case where the keyword isn't found
    # Expand the context around the keyword
    start = max(0, start_index - window)
    end = min(len(full_text), start_index + window)
    return full_text[start:end]


def main(pdf_path, question, keyword):
    # Extract text from the entire document
    full_text = extract_text_from_pdf(pdf_path)
    # Get a slice of the text relevant to the question
    context = get_relevant_context(full_text, keyword)
    # Answer the question
    answer = answer_question(question, context)
    return answer


# Example usage
pdf_path = 'upsc_full_doc.pdf'
question = 'What are the age limits for a candidate who belongs to a Scheduled Caste or a Scheduled Tribe '
keywords = 'age limits'  # This helps to narrow down the section of the PDF to use

print("Answer:", main(pdf_path, question, keywords))


Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Answer: up to a maximum of five years


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
import PyPDF2

def extract_text_from_pdf(pdf_path):
  """
  Extracts text content from a PDF document.

  Args:
      pdf_path: Path to the PDF document.

  Returns:
      A string containing the extracted text content.
  """
  with open(pdf_path, 'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    full_text = ""
    for page in pdf_reader.pages:
      full_text += page.extract_text()
    return full_text


def load_qa_model():
  """
  Loads the pretrained question answering model and tokenizer.

  Returns:
      A tuple containing the question answering pipeline and tokenizer.
  """
  model_name = "deepset/roberta-base-squad2"  # Example model for Q&A
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForQuestionAnswering.from_pretrained(model_name)
  qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)
  return qa_pipeline, tokenizer


def answer_question(pdf_path, question):
  """
  Answers a question based on the provided PDF document.

  Args:
      pdf_path: Path to the PDF document.
      question: The question to be answered.

  Returns:
      A dictionary containing the answer text, answer start/end positions, and score.
  """
  qa_pipeline, tokenizer = load_qa_model()
  text = extract_text_from_pdf(pdf_path)

  # Encode the question and context for the model
  inputs = tokenizer(question, text, return_tensors="pt")

  # Get the answer using the pipeline
  answer = qa_pipeline(inputs)
  return answer


In [26]:
import PyPDF2
import pandas as pd


import torch
from transformers import BertTokenizer, BertForQuestionAnswering

# Example of using a different model
model_name = 'deepset/bert-base-cased-squad2'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForQuestionAnswering.from_pretrained(model_name)


def answer_question(question, context):
    inputs = tokenizer.encode_plus(question, context, add_special_tokens=True, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        answer_start_scores = outputs.start_logits
        answer_end_scores = outputs.end_logits

    print("Start Scores:", answer_start_scores)  # Debugging line
    print("End Scores:", answer_end_scores)  # Debugging line

    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1

    if answer_start == 0 or answer_end == 1:
        return "No valid answer found in the text"

    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][answer_start:answer_end]))
    return answer



def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as pdf_file:
        reader = PyPDF2.PdfReader(pdf_file)
        title = reader.metadata.title
        text = ""
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text()
        return text
        
        
def get_relevant_context(full_text, keywords, window=500):
    min_index = float('inf')
    selected_context = ""
    for keyword in keywords:
        start_index = full_text.lower().find(keyword.lower())
        if start_index != -1 and start_index < min_index:
            min_index = start_index
            start = max(0, start_index - window)
            end = min(len(full_text), start_index + window)
            selected_context = full_text[start:end]
    
    if selected_context == "":
        return "Keywords not found in the text"  # Handle the case where no keywords are found
    return selected_context


def main(pdf_path, question, keyword):
    # Extract text from the entire document
    full_text = extract_text_from_pdf(pdf_path)
    # Get a slice of the text relevant to the question
    context = get_relevant_context(full_text, keywords)
    # Answer the question
    answer = answer_question(question, context)
    return answer


pdf_path = 'upsc_full_doc.pdf'
question = 'What are the age limits for a candidate who belongs to a Scheduled Caste or a Scheduled Tribe?'
keywords = ['age limits', 'Scheduled Caste', 'Scheduled Tribe']  # Broaden the scope of search to ensure completeness

print("Answer:", main(pdf_path, question, keywords))


Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Start Scores: tensor([[ 0.5027, -4.6956, -5.8340, -5.4492, -5.9666, -7.2986, -7.4106, -7.4088,
         -8.1165, -8.3776, -8.4073, -8.1699, -7.9511, -6.4104, -8.2450, -8.5787,
         -8.0321, -6.8370, -8.2482, -9.1531,  0.5027, -7.8872, -8.1108, -8.6063,
         -8.6845, -6.9351, -7.6149, -8.4648, -8.6348, -8.6048, -7.2779, -7.6035,
         -8.8975, -6.5865, -8.8047, -8.2605, -9.1181, -8.1452, -8.9077, -6.5818,
         -7.8905, -8.5770, -8.7645, -8.8118, -7.1710, -7.6168, -9.0973, -7.1825,
         -8.5274, -9.0562, -7.6151, -8.6691, -7.4621, -8.6670, -8.7460, -8.7407,
         -7.5994, -8.7035, -8.8073, -7.1613, -8.7709, -7.6904, -9.1087, -9.0596,
         -8.3421, -9.1889, -9.0173, -8.8530, -8.1699, -8.9989, -9.1138, -8.9341,
         -8.9141, -8.6602, -8.3794, -6.7713, -7.7193, -8.5538, -8.5707, -8.5126,
         -7.6871, -8.2160, -8.9874, -7.4277, -7.2929, -7.9644, -7.8257, -6.5190,
         -6.2891, -4.5392, -6.8735, -7.1003, -8.7192, -7.6685, -6.8547, -7.6935,
         -8.76

In [21]:
import PyPDF2
import pandas as pd


import torch
from transformers import BertTokenizer, BertForQuestionAnswering

# Initialize tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

# Function to answer a question given a context
def answer_question(question, context):
    # Encode the question and context for the model
    inputs = tokenizer(question, context, add_special_tokens=True, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs["input_ids"].tolist()[0]

    # Predict the start and end positions of the answer
    with torch.no_grad():
        outputs = model(**inputs)
        answer_start_scores = outputs.start_logits
        answer_end_scores = outputs.end_logits

    # Decode the predicted answer
    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1
    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end]))

    return answer


def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as pdf_file:
        reader = PyPDF2.PdfReader(pdf_file)
        title = reader.metadata.title
        text = ""
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text()
        return text
        
        
def get_relevant_context(full_text, keywords, window=1000):
    min_index = len(full_text)
    selected_context = ""
    
    for keyword in keywords:
        start_index = full_text.lower().find(keyword.lower())
        if start_index == -1:
            continue  # Skip if keyword is not found
        # Expand the context around the keyword
        start = max(0, start_index - window)
        end = min(len(full_text), start_index + window)
        if start < min_index:
            min_index = start  # Update the earliest start found
            selected_context = full_text[start:end]  # Update the context based on the closest keyword found
    
    if selected_context == "":
        return "Keywords not found in the text"  # Handle the case where no keywords are found
    
    return selected_context


def main(pdf_path, question, keyword):
    # Extract text from the entire document
    full_text = extract_text_from_pdf(pdf_path)
    # Get a slice of the text relevant to the question
    context = get_relevant_context(full_text, keyword)
    # Answer the question
    answer = answer_question(question, context)
    return answer


pdf_path = 'upsc_full_doc.pdf'
question = 'What are the age limits for a candidate who belongs to a Scheduled Caste or a Scheduled Tribe?'
keywords = ['age limits', 'Scheduled Caste', 'Scheduled Tribe']  # Broaden the scope of search to ensure completeness

print("Answer:", main(pdf_path, question, keywords))



Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Answer: [CLS]
